# Task 2 — Step 1: Standard Proximal Policy Optimization (PPO) Continuation

### Objective & Mathematical Formulation
Online RLHF optimizes policy $\pi_\theta$ against a frozen reward model with a reference-policy penalty:
$$r_t = r_{\text{task}}\cdot \mathbf{1}[t = T] - \beta_{\text{KL}} \left(\log \pi_\theta(a_t|s_t) - \log \pi_{\text{ref}}(a_t|s_t)\right)$$

The PPO clipped surrogate objective updates the policy:
$$\mathcal{L}_{\text{clip}}(\theta) = \hat{\mathbb{E}}_t \left[ \min\left( \rho_t(\theta) \hat{A}_t, \text{clip}(\rho_t(\theta), 1-\epsilon, 1+\epsilon) \hat{A}_t \right) \right]$$
where importance ratio $\rho_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\text{old}}(a_t|s_t)}$, and $\hat{A}_t$ is estimated using Generalized Advantage Estimation (GAE) with a learned value critic $V_\phi(s)$.

### Protocol
- Continue from the course-provided staff midpoint checkpoints (`checkpoints/ppo_midpoint_policy`, `checkpoints/ppo_midpoint_value`).
- Execute 20 updates.
- Track trajectories of reward, KL drift, value loss, policy loss, clip fraction, entropy, gradient norm, response length, peak VRAM, and wall-clock runtime.


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Run PPO Continuation or Load Saved Logs


In [ ]:
FORCE_RUN = False
if FORCE_RUN:
    !python -m task2_ppo.continue_train --config configs/ppo.yaml --run-name standard
    !python -m task2_ppo.evaluate --config configs/ppo.yaml --adapter outputs/task2_ppo/standard --name standard
else:
    print("[OK] Loading existing PPO continuation artifacts...")


## 2. Summary Table of PPO Continuation Run


In [ ]:
summary_file = "results/task2_ppo/ppo_summary_standard.json"
if os.path.exists(summary_file):
    with open(summary_file, encoding='utf-8') as f:
        summary_data = json.load(f)
    display(pd.DataFrame([summary_data]))
else:
    print("Run continuation to generate ppo_summary_standard.json")


## 3. Visualizing PPO Continuation Trajectories (20 Updates)


In [ ]:
log_file = "results/task2_ppo/ppo_train_standard.jsonl"
if os.path.exists(log_file):
    records = [json.loads(line) for line in open(log_file, encoding='utf-8')]
    df = pd.DataFrame(records)
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    
    # 1. Reward
    axes[0, 0].plot(df['update'], df['reward_mean'], color='#10B981', lw=2)
    axes[0, 0].set_title('Mean Learned Reward')
    axes[0, 0].set_xlabel('Update')
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. KL Drift
    axes[0, 1].plot(df['update'], df['kl_mean'], color='#EF4444', lw=2)
    axes[0, 1].set_title('Mean Policy Drift (KL)')
    axes[0, 1].set_xlabel('Update')
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. Clip Fraction
    axes[0, 2].plot(df['update'], df['clip_fraction']*100, color='#F59E0B', lw=2)
    axes[0, 2].set_title('Token Clip Fraction (%)')
    axes[0, 2].set_xlabel('Update')
    axes[0, 2].grid(True, alpha=0.3)
    
    # 4. Value Loss
    axes[1, 0].plot(df['update'], df['value_loss'], color='#8B5CF6', lw=2)
    axes[1, 0].set_title('Critic Value Loss')
    axes[1, 0].set_xlabel('Update')
    axes[1, 0].grid(True, alpha=0.3)
    
    # 5. Policy Entropy
    axes[1, 1].plot(df['update'], df['entropy'], color='#06B6D4', lw=2)
    axes[1, 1].set_title('Policy Entropy')
    axes[1, 1].set_xlabel('Update')
    axes[1, 1].grid(True, alpha=0.3)
    
    # 6. Response Length
    axes[1, 2].plot(df['update'], df['mean_response_length'], color='#6366F1', lw=2)
    axes[1, 2].set_title('Response Length (tokens)')
    axes[1, 2].set_xlabel('Update')
    axes[1, 2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


## 4. Compute Footprint Analysis
Report peak VRAM and wall-clock runtime for the standard 20-update continuation to compare later against critic-free GRPO in Task 3.
